In [20]:
from openai import OpenAI
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [21]:
load_dotenv()
openai_client = OpenAI()

In [22]:
INSTRUCTIONS = '''
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
'''

### Loading data

In [23]:
reader = GithubRepositoryDataReader(
        repo_owner="DataTalksClub",
        repo_name="llm-zoomcamp",
        commit_id="8c1834d",
        allowed_extensions={"md"},
        filename_filter=lambda path: "/lessons/" in path,
    )

files = reader.read()

documents = []

# dict_keys(['content', 'filename'])
for file in files:
    doc = file.parse()
    documents.append(doc)

chunks = chunk_documents(documents, size=2000, step=1000)

### Indexing the data

In [24]:
index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
index.fit(documents)

### Defining search tool for the agent

In [25]:
def search(query: str, num_results: int=5):
    """
    Search the index for documents matching the query.

    Args:
        query (str): The search query.
        num_results (int): The number of search results to return. Default is 5.
    """
    boost_dict = {'content': 1.0}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict,
    )

### Initializing the agent

In [26]:
agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [27]:
runner = OpenAIResponsesRunner(
        tools=agent_tools,
        developer_prompt=INSTRUCTIONS,
        chat_interface=chat_interface,
        llm_client=OpenAIClient(model="gpt-5.4-mini")
    )

In [ ]:
runner.run()

You: How does the agentic loop work, and how is it different from plain RAG?


-> Response received


-> Response received
